# 🔥 SDXL Image Forge — Google Colab

**Antes de correr:** Ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`

Celdas en orden:
1. Instalar dependencias
2. Crear archivos del proyecto
3. Cargar modelo + arrancar FastAPI
4. Exponer con ngrok → URL pública

## Celda 1 — Instalar dependencias

In [ ]:
!pip install -q fastapi uvicorn[standard] diffusers transformers accelerate \
               safetensors Pillow pyngrok nest-asyncio python-multipart pydantic

## Celda 2 — Crear estructura de archivos

Esto escribe `main.py` y `static/index.html` directo en Colab.
No necesitas subir nada manualmente.

In [ ]:
import os
os.makedirs('static', exist_ok=True)

# ── main.py ──────────────────────────────────────────────────────────────
main_py = '''
import os, io, base64, logging, torch
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional
from diffusers import DiffusionPipeline

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE    = torch.float16 if DEVICE == "cuda" else torch.float32
pipe     = None

@asynccontextmanager
async def lifespan(app):
    global pipe
    logger.info(f"Cargando SDXL en {DEVICE}...")
    pipe = DiffusionPipeline.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE,
        use_safetensors=True, variant="fp16" if DEVICE=="cuda" else None
    ).to(DEVICE)
    if DEVICE == "cuda":
        pipe.enable_attention_slicing()
        pipe.enable_vae_slicing()
    logger.info("Modelo listo.")
    yield
    del pipe
    if DEVICE == "cuda": torch.cuda.empty_cache()

app = FastAPI(title="SDXL Image Forge", version="2.0.0", lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/static", StaticFiles(directory="static"), name="static")

class GenerateRequest(BaseModel):
    prompt: str                  = Field(..., min_length=3, max_length=500)
    negative_prompt: Optional[str] = Field(default="cartoon, anime, low quality, blurry, watermark")
    guidance_scale: float        = Field(default=7.5, ge=0.0, le=20.0)
    num_inference_steps: int     = Field(default=30, ge=1, le=50)
    seed: Optional[int]          = Field(default=None)
    width: int                   = Field(default=1024, ge=512, le=1024)
    height: int                  = Field(default=1024, ge=512, le=1024)

class GenerateResponse(BaseModel):
    image_base64: str
    prompt_used: str
    parameters: dict

@app.get("/", include_in_schema=False)
async def root(): return FileResponse("static/index.html")

@app.get("/health")
async def health():
    return {
        "status": "ok", "model_loaded": pipe is not None,
        "device": DEVICE, "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
    }

@app.post("/generate", response_model=GenerateResponse)
async def generate_image(req: GenerateRequest):
    if pipe is None:
        raise HTTPException(503, "Modelo cargando, espera unos segundos.")
    generator = torch.Generator(device=DEVICE).manual_seed(req.seed) if req.seed else None
    logger.info(f"Generando: {req.prompt[:60]}")
    try:
        with torch.inference_mode():
            result = pipe(
                prompt=req.prompt, negative_prompt=req.negative_prompt,
                guidance_scale=req.guidance_scale, num_inference_steps=req.num_inference_steps,
                width=req.width, height=req.height, generator=generator
            )
        buf = io.BytesIO()
        result.images[0].save(buf, format="PNG")
        buf.seek(0)
        return GenerateResponse(
            image_base64=base64.b64encode(buf.read()).decode(),
            prompt_used=req.prompt,
            parameters={
                "guidance_scale": req.guidance_scale,
                "num_inference_steps": req.num_inference_steps,
                "negative_prompt": req.negative_prompt,
                "seed": req.seed, "width": req.width,
                "height": req.height, "device": DEVICE
            }
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise HTTPException(500, "GPU sin memoria. Reduce steps o resolución.")
    except Exception as e:
        logger.exception("Error")
        raise HTTPException(500, str(e))

@app.get("/presets")
async def get_presets():
    return {"presets": [
        {"style": "Fantasía Oscura",
         "prompt": "burned medieval knight in full plate armor, charred black metal armor cracked with glowing embers, lava-like fissures emitting orange and red light, tattered cape disintegrating into ash, apocalyptic yellow sky, ruined gothic castle, cinematic dark fantasy atmosphere, photorealistic, masterpiece",
         "negative_prompt": "cartoon, anime, low poly, plastic armor, bright colors, blurry, watermark"},
        {"style": "Ciencia Ficción",
         "prompt": "futuristic cyberpunk city at night, neon lights reflecting on wet streets, flying cars, holographic advertisements, dense fog, ultra detailed, cinematic lighting, photorealistic, 8k",
         "negative_prompt": "cartoon, anime, low quality, blurry, watermark, daylight"},
        {"style": "Naturaleza Épica",
         "prompt": "majestic snow-capped mountain at golden hour, dramatic clouds, crystal clear lake reflection, ancient pine forest, volumetric light rays, ultra realistic landscape photography, sharp focus, masterpiece",
         "negative_prompt": "cartoon, people, buildings, text, watermark, blurry"},
        {"style": "Retrato",
         "prompt": "cinematic portrait of a wise old explorer, weathered face with deep wrinkles, piercing blue eyes, dramatic side lighting, shallow depth of field, film grain, Kodak Portra 400, photorealistic",
         "negative_prompt": "cartoon, anime, painting, illustration, low quality, watermark"},
    ]}
'''

with open('main.py', 'w') as f:
    f.write(main_py)

print('✅ main.py creado')

In [ ]:
# ── static/index.html ─────────────────────────────────────────────────────
# Lee el HTML desde Drive si lo tienes ahí, o pégalo directo.
# OPCIÓN A: copiar desde Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
# Cambia esta ruta a donde tengas tu index.html en Drive
shutil.copy('/content/drive/MyDrive/sdxl-app/static/index.html', 'static/index.html')
print('✅ index.html copiado desde Drive')

In [ ]:
# OPCIÓN B: si NO tienes Drive, descarga el HTML desde GitHub
# Cambia la URL a tu repo cuando lo subas
# !wget -q -O static/index.html https://raw.githubusercontent.com/TU_USUARIO/sdxl-app/main/static/index.html
# print('✅ index.html descargado desde GitHub')

## Celda 3 — Verificar GPU

In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM total:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  Sin GPU — ve a Entorno de ejecución → Cambiar tipo → T4 GPU')

## Celda 4 — Arrancar FastAPI + exponer con ngrok

El modelo se carga aquí (~2-3 min). Cuando aparezca la URL pública, ya puedes usarla.

In [ ]:
import nest_asyncio, uvicorn, threading
from pyngrok import ngrok

nest_asyncio.apply()  # Permite correr asyncio dentro de Colab

PORT = 8000

# Arranca uvicorn en un hilo separado para no bloquear el notebook
def run_server():
    uvicorn.run('main:app', host='0.0.0.0', port=PORT, log_level='info')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Espera a que el servidor arranque
import time
time.sleep(3)

# Exponer con ngrok (NO necesitas cuenta para pruebas cortas)
public_url = ngrok.connect(PORT)
print('\n' + '='*50)
print(f'🔥  App pública: {public_url}')
print(f'📖  Swagger UI:  {public_url}/docs')
print(f'🏥  Health:      {public_url}/health')
print('='*50)
print('\n⏳ El modelo está cargando en background (~2-3 min).')
print('Mientras carga, /health mostrará model_loaded: false.')
print('Cuando diga model_loaded: true, ya puedes generar imágenes.')

## Celda 5 (opcional) — Token de ngrok para sesiones más largas

Sin token ngrok las URLs expiran en ~2 horas y no puedes tener
múltiples túneles. Con cuenta gratis en ngrok.com obtienes URLs
que duran toda la sesión de Colab.

In [ ]:
# Crea cuenta gratis en https://ngrok.com y copia tu authtoken
# Descomenta y pega tu token:

# from pyngrok import ngrok
# ngrok.set_auth_token('TU_NGROK_TOKEN_AQUI')
# print('✅ Token ngrok configurado')

## Celda 6 (opcional) — Probar la API directo desde el notebook

In [ ]:
import requests, base64
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

# Prueba rápida contra el servidor local
resp = requests.post('http://localhost:8000/generate', json={
    'prompt': 'burned medieval knight in full plate armor, glowing ember cracks, apocalyptic sky, photorealistic, masterpiece',
    'negative_prompt': 'cartoon, anime, blurry, watermark',
    'guidance_scale': 7.5,
    'num_inference_steps': 30,
    'seed': 42
}, timeout=180)

if resp.status_code == 200:
    data = resp.json()
    img  = Image.open(BytesIO(base64.b64decode(data['image_base64'])))
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(data['prompt_used'][:80])
    plt.show()
    print('Parámetros:', data['parameters'])
else:
    print('Error:', resp.status_code, resp.json())